In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [3]:
import json
import torch 
from src.config import CHECKPOINT_DIR
from scripts.common.get_device import get_available_device
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.config import DEFAULT_STGCN_PARAMS_V1, POSE_DATASET_ROOT_OCSORT2, DATASET_ROOT
from src.Skeleton_model.stgcn import STGCN
from src.XAI.STGCN_grad_cam import STGCNGradCam

In [4]:
experiment_root = CHECKPOINT_DIR / "STGCN_V1" / "final_seed_search_updated_masked_bn" / "final_seed42"
device = get_available_device()

with open(experiment_root / "config.json", "r") as f:
    hyperparameters = json.load(f)

train_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT_OCSORT2, split="train", max_people=hyperparameters["max_people"])
val_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT_OCSORT2, split="val", max_people=hyperparameters["max_people"])
rgb_dataset = RWF2000Dataset(DATASET_ROOT, split="val", return_rgb_frames=True, num_frames=150)
radii = compute_joint_distance_to_center_of_gravity(train_dataset)
skeleton_graph = SkeletonGraph(radii, normalisation=hyperparameters["adjacency_normalisation_mode"])
model = STGCN(skeleton_graph.A, temporal_kernel_size=hyperparameters["temporal_kernel_size"], dropout=hyperparameters["dropout"], edge_importance_weighting=hyperparameters["edge_importance_weighting"]).to(device)

checkpoint = torch.load(experiment_root / "best_model.pt", map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

Using cuda:4 with 22.94 GB free


/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


STGCN(
  (data_batch_norm): MaskedBatchNorm1d(51, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (stgcn_blocks): ModuleList(
    (0): STGCNBlock(
      (spatial_graph_conv): SpatialGraphConv(
        (channel_transform): Conv2d(3, 192, kernel_size=(1, 1), stride=(1, 1))
      )
      (bn1): MaskedBatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (temporal_conv): Sequential(
        (0): ReLU(inplace=True)
        (1): Conv2d(64, 64, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0), bias=False)
      )
      (bn2): MaskedBatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0, inplace=False)
      (relu): ReLU(inplace=True)
    )
    (1-3): 3 x STGCNBlock(
      (spatial_graph_conv): SpatialGraphConv(
        (channel_transform): Conv2d(64, 192, kernel_size=(1, 1), stride=(1, 1))
      )
      (bn1): MaskedBatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

In [5]:
stg_gradcam = STGCNGradCam(
    model=model,
    target_layer=model.stgcn_blocks[-1],
    normalisation_mode="per_video",
)
video_num = 7

input_tensor, label = val_dataset[video_num]
input_tensor = input_tensor.unsqueeze(0).to(device)

heatmap = stg_gradcam.generate_heatmap(input_tensor)

In [7]:
import torch.nn.functional as F
pre_zero_total, pre_count_total = 0, 0
post_zero_total, post_count_total = 0, 0
post_detected_zero_total, post_detected_count_total = 0, 0
skipped = 0

for idx in range(len(val_dataset)):
    input_tensor, label = val_dataset[idx]
    input_tensor = input_tensor.unsqueeze(0).to(device)

    post_cam = stg_gradcam.generate_heatmap(input_tensor)  # [T, V, M]

    if post_cam.sum() == 0:
        skipped += 1
        continue

    activations = stg_gradcam.activations
    gradients = stg_gradcam.gradients
    weights = gradients.mean(dim=(2, 3), keepdim=True)
    pre_cam = (weights * activations).sum(dim=1)
    pre_cam = F.relu(pre_cam)
    pre_cam = pre_cam.permute(1, 2, 0)  # [reduced_time, V, M]

    pre_zero_total += (pre_cam == 0).sum().item()
    pre_count_total += pre_cam.numel()
    post_zero_total += (post_cam == 0).sum().item()
    post_count_total += post_cam.numel()

    # detected joints only, matching Table 4.5's scope
    detected = input_tensor[0, 2] > 0  # [T, V, M] confidence channel, adjust indexing if needed
    post_cam_detected = post_cam[detected]
    post_detected_zero_total += (post_cam_detected == 0).sum().item()
    post_detected_count_total += post_cam_detected.numel()

print(f"Skipped {skipped} empty clips")
print(f"Pre-interpolation (all cells):        {100 * pre_zero_total / pre_count_total:.1f}% zero")
print(f"Post-interpolation (all cells):       {100 * post_zero_total / post_count_total:.1f}% zero")
print(f"Post-interpolation (detected only):   {100 * post_detected_zero_total / post_detected_count_total:.1f}% zero")

Skipped 52 empty clips
Pre-interpolation (all cells):        56.4% zero
Post-interpolation (all cells):       53.7% zero
Post-interpolation (detected only):   37.9% zero
